# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamzanPUCIT/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

---
**Contract in plain words (5 answers):**
1. **One row** = one content page (pseudonymized `content_id`), one trailing-90-day snapshot.
2. **Table:** starter release `content_refresh_anonymized.csv` (queried with DuckDB below). *Note: the gated
   Hugging Face warehouse was not reachable from my setup, so the three proofs run on the starter release with the
   same query pattern; the warehouse month=2026-03 re-run is a drop-in change of the FROM clause.*
3. **Time window:** inputs = days 31–60 back (`*_prev_30d`) + static properties; outcome = days 1–30 back.
4. **Label / proxy:** `demand_drop` = last-30d impressions < 0.8 × previous-30d, pages with ≥ 100 earlier impressions.
5. **Deliberately excluded:** `trend_direction` / `trend_pct` and every `*_last_30d` column (they ARE the label window).

In [1]:
# Setup: load the starter dataset (works in a local clone and in Colab)
from pathlib import Path
import numpy as np
import pandas as pd

RAW_URL = ("https://raw.githubusercontent.com/RamzanPUCIT/flyrank-ml-internship/"
           "main/data/raw/content_refresh_anonymized.csv")
local = [p for p in [Path("../../data/raw/content_refresh_anonymized.csv"),
                     Path("data/raw/content_refresh_anonymized.csv")] if p.exists()]
df = pd.read_csv(local[0] if local else RAW_URL)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("rows x cols:", df.shape)

import duckdb
con = duckdb.connect()
con.register("pages", df)
print(con.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT content_id) AS pages, COUNT(DISTINCT client_id) AS clients FROM pages").df())

rows x cols: (30000, 44)


    rows  pages  clients
0  30000  30000       32


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

---
| Bucket | Fields |
|---|---|
| **Feature** (5, knowable at decision moment) | see the five-feature frame in section 3 |
| **Label** | `demand_drop` (built from `impressions_last_30d` vs `impressions_prev_30d`) |
| **Context** (never features) | `content_id`, `client_id` (grouping/splits only) |
| **Excluded** | `trend_*`, `*_last_30d`, 90-day totals/averages (overlap label window), `days_since_last_update`, tiers, `provider_used`, `model_used` |

In [2]:
FEATURES = {
  "impressions_prev_30d": "knowable because it is measured in days 31-60, before the outcome window",
  "ctr_prev":             "knowable because clicks/impressions of days 31-60 are both closed before the outcome",
  "content_age_days":     "knowable because the publish date is fixed long before the outcome window",
  "word_count":           "knowable because it is a property of the published page",
  "search_volume":        "knowable because it is keyword-research context, not measured in the outcome window"}
pd.Series(FEATURES, name="available when?")

impressions_prev_30d    knowable because it is measured in days 31-60,...
ctr_prev                knowable because clicks/impressions of days 31...
content_age_days        knowable because the publish date is fixed lon...
word_count              knowable because it is a property of the publi...
search_volume           knowable because it is keyword-research contex...
Name: available when?, dtype: str

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

---
Three proofs (outputs below): **(1) grain** — one row per `content_id`; **(2) slice size and windows** — row count,
clients, and age span of my slice; **(3) availability** — pages with earlier-window demand, filtered `IS TRUE`.
Then the five-feature frame, and **the trap**: adding one label-derived column makes the score jump to near
perfect — I show it, then delete it and keep the honest number.

In [3]:
# Proof 1 - grain
print(con.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT content_id) AS distinct_pages, COUNT(*) = COUNT(DISTINCT content_id) AS one_row_per_page FROM pages").df())
# Proof 2 - slice row count and span
print(con.sql("""SELECT COUNT(*) AS slice_rows, COUNT(DISTINCT client_id) AS clients,
                 MIN(content_age_days) AS youngest_days, MAX(content_age_days) AS oldest_days
                 FROM pages WHERE impressions_prev_30d >= 100""").df())
# Proof 3 - availability with IS TRUE
print(con.sql("""SELECT COUNT(*) AS all_rows,
                 COUNT(*) FILTER (WHERE (impressions_prev_30d >= 100) IS TRUE) AS rows_with_demand,
                 COUNT(*) FILTER (WHERE (word_count IS NOT NULL) IS TRUE) AS rows_with_word_count
                 FROM pages""").df())

# Five-feature frame + quick score, then the trap
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
# My lane slice (same definition in every notebook)
MIN_PREV_IMPR = 100          # page needs real demand in the earlier window
lane = df[df["impressions_prev_30d"] >= MIN_PREV_IMPR].copy()
# proxy target: observed later-window outcome (last 30d vs the 30d before)
lane["demand_drop"] = (lane["impressions_last_30d"] < 0.8 * lane["impressions_prev_30d"]).astype(int)
lane["ctr_prev"] = lane.clicks_prev_30d / lane.impressions_prev_30d * 100
F = lane[list(FEATURES)].fillna(0)
y, g = lane.demand_drop, lane.client_id
rf = RandomForestClassifier(150, min_samples_leaf=20, random_state=42, n_jobs=-1)
honest = cross_val_score(rf, F, y, groups=g, cv=GroupKFold(5), scoring="roc_auc").mean()
F_leak = F.assign(trend_pct=lane["trend_pct"])          # label-derived column, added ON PURPOSE
leaky = cross_val_score(rf, F_leak, y, groups=g, cv=GroupKFold(5), scoring="roc_auc").mean()
F_leak = F_leak.drop(columns="trend_pct")               # deleted again
print(f"honest 5-feature AUC: {honest:.3f} | with trend_pct (leak): {leaky:.3f} -> removed, keep {honest:.3f}")

    rows  distinct_pages  one_row_per_page
0  30000           30000              True
   slice_rows  clients  youngest_days  oldest_days
0       18010       30             90          564
   all_rows  rows_with_demand  rows_with_word_count
0     30000             18010                 22301


honest 5-feature AUC: 0.590 | with trend_pct (leak): 1.000 -> removed, keep 0.590


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

---
**Named limitation:** one snapshot only — inputs and outcome are two adjacent 30-day windows of the same export,
so there is no calendar-time holdout, and a 30-day change can be seasonal. Also two clients hold about half the
slice, so results lean toward their sites.

In [4]:
share = lane.client_id.value_counts(normalize=True).head(2)
print("share of slice held by the two largest clients:", round(share.sum(), 3))

share of slice held by the two largest clients: 0.502


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.